# Library Imports

In [3]:
# TimeCopilot Imports
from timecopilot import TimeCopilotForecaster
from timecopilot.models.foundation.tabpfn import TabPFN

In [ ]:
!pip install utilsforecast
import pandas as pd
import matplotlib.pyplot as plt
from utilsforecast.plotting import plot_series
from utilsforecast.losses import bias, rmse, mae, mape, bias
from utilsforecast.evaluation import evaluate

# Data Preparation (this is shortened from first version)

In [ ]:
df_base = pd.read_parquet('/Users/emahedman/Downloads/ISA444/FinalProject/Data/sample_hotels-1.parquet')
df_base.head()

,unique_id,ds,holiday_flag,target_day,target_month,target_year,location_type,hotel_type,y,otb_1,...,otb_51,otb_52,otb_53,otb_54,otb_55,otb_56,otb_57,otb_58,otb_59,otb_60
1430,hotel_0,2022-01-01,no,Sat,Jan,2022,NonSuburban,Resorts & Destinations,0.975309,0.679012,...,0.197531,0.197531,0.197531,0.185185,0.160494,0.160494,0.160494,0.160494,0.160494,0.160494
1431,hotel_0,2022-01-02,no,Sun,Jan,2022,NonSuburban,Resorts & Destinations,0.493827,0.308642,...,0.074074,0.074074,0.074074,0.074074,0.074074,0.061728,0.061728,0.061728,0.061728,0.049383
1432,hotel_0,2022-01-03,no,Mon,Jan,2022,NonSuburban,Resorts & Destinations,0.456790,0.358025,...,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691
1433,hotel_0,2022-01-04,no,Tue,Jan,2022,NonSuburban,Resorts & Destinations,0.592593,0.419753,...,0.074074,0.074074,0.074074,0.074074,0.061728,0.061728,0.037037,0.037037,0.024691,0.024691
1434,hotel_0,2022-01-05,no,Wed,Jan,2022,NonSuburban,Resorts & Destinations,0.530864,0.407407,...,0.074074,0.074074,0.074074,0.074074,0.074074,0.049383,0.049383,0.024691,0.024691,0.012346


Drop unimformative variable

In [18]:
df_base = df_base.drop(columns=['target_year'])

Dummies

In [19]:
 # Convert holiday flag to boolean manually
 df_base['holiday_flag'] = df_base['holiday_flag'].astype('bool')


In [20]:
cat_cols = [
    'target_day',
    'target_month',
    'location_type',
    'hotel_type'
]

for col in cat_cols:
    df_base[col] = df_base[col].astype('category')

In [21]:
df_base = pd.get_dummies(df_base, columns=cat_cols, drop_first=True)

OTB

In [22]:
# Drop all OTB values before otb_28 because that information wouldn't actually be available at the time of forecasting
# By only keeping otb_28 and beyond, I ensure the model doesn't "cheat" by looking at data from inside the 28-day period
columns_to_drop = [f'otb_{i}' for i in range(1, 28)]
df_base = df_base.drop(columns=columns_to_drop)
display(df_base.head())

,unique_id,ds,holiday_flag,y,otb_28,otb_29,otb_30,otb_31,otb_32,otb_33,...,target_month_Jun,target_month_Mar,target_month_May,target_month_Nov,target_month_Oct,target_month_Sep,location_type_NonSuburban,hotel_type_Key Central Business District,hotel_type_Other High Leisure Mix,hotel_type_Resorts & Destinations
1430,hotel_0,2022-01-01,True,0.975309,0.296296,0.283951,0.296296,0.296296,0.283951,0.259259,...,False,False,False,False,False,False,True,False,False,True
1431,hotel_0,2022-01-02,True,0.493827,0.086420,0.086420,0.086420,0.086420,0.086420,0.086420,...,False,False,False,False,False,False,True,False,False,True
1432,hotel_0,2022-01-03,True,0.456790,0.061728,0.061728,0.061728,0.061728,0.061728,0.061728,...,False,False,False,False,False,False,True,False,False,True
1433,hotel_0,2022-01-04,True,0.592593,0.111111,0.111111,0.111111,0.111111,0.098765,0.098765,...,False,False,False,False,False,False,True,False,False,True
1434,hotel_0,2022-01-05,True,0.530864,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,...,False,False,False,False,False,False,True,False,False,True


Drop Hotels

In [23]:
hotels_to_drop = ['hotel_28', 'hotel_77']
df_base = df_base[df_base['unique_id'].isin(hotels_to_drop) == False]

Test/Train and Pred/No Pred Split

In [24]:
# cutoff
cutoff = '2023-05-31'

#train/test split
train = df_base[df_base['ds'] <= cutoff]
test = df_base[df_base['ds'] > cutoff]

# full df no pred
df_no_pred = df_base[['unique_id', 'ds', 'y']]

# train
train_base = train[['unique_id', 'ds', 'y']] # Nixtila format dataset
train_ml = train.copy() # full dataset for ML

test_base = test[['unique_id', 'ds', 'y']]
test_ml = test.copy()

# TabPFN

In [31]:
# Establish the foundation model object
tabpfn = TabPFN(
    features=None,
    context_length=365, # 1 year lookback period
    alias="TabPFN"
)

# Instantiate orchestrator
copilot = TimeCopilotForecaster(
    models=[tabpfn]
)

In [37]:
# Run 5-fold cross-validation
tabpfn_cv = copilot.cross_validation(
    df=df_no_pred,
    h=28,
    n_windows=5,
    step_size=28
)

Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:01<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Predicting time series: 100%|██████████| 17/17 [00:32<00:00,  1.94s/it]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|██████████| [00:00<00:00]
Processing: 100%|████████

In [39]:
eval_tabpfn = evaluate(
    df = tabpfn_cv,
    metrics = [mape, rmse, mae, bias],
    models = ['TabPFN']
)

In [40]:
eval_tabpfn

,unique_id,cutoff,metric,TabPFN
0,hotel_0,2023-02-10,mape,0.110455
1,hotel_105,2023-02-10,mape,0.104526
2,hotel_112,2023-02-10,mape,0.126427
3,hotel_126,2023-02-10,mape,0.098166
4,hotel_133,2023-02-10,mape,0.218853
...,...,...,...,...
335,hotel_7,2023-06-02,bias,0.148535
336,hotel_70,2023-06-02,bias,-0.041435
337,hotel_84,2023-06-02,bias,-0.038702
338,hotel_91,2023-06-02,bias,-0.014688


In [42]:
# save as CSV
eval_tabpfn.to_csv('/Users/emahedman/Downloads/ISA444/FinalProject/Evaluations/eval_tabpfn.csv', index=False)